In [16]:
#NAIVE RAG using Llama Index

# %pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [17]:
#configuration
Settings.llm = Ollama(model="gemma4:e4b", temperature=0,
    request_timeout=600)
Settings.embed_model = OllamaEmbedding(model_name="qwen3-embedding:0.6b")
Settings.chunk_size_limit = 512
Settings.chunk_overlap = 50

In [18]:
#document loading

from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("data/").load_data()

In [19]:
#main

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex

# 1. Parse into nodes explicitly
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

# 2. Build the index from nodes (not from raw documents)
index = VectorStoreIndex(nodes)

# 3. Build retriever explicitly (this is the piece that changes across architectures)
retriever = index.as_retriever(similarity_top_k=5)

# 4. Build query engine from the retriever
from llama_index.core.query_engine import RetrieverQueryEngine
query_engine = RetrieverQueryEngine.from_args(retriever)

In [20]:
#Query

response = query_engine.query("can u summerize this paper")
print(response.response)
print([n.text for n in response.source_nodes])  # this is your `contexts` field for RAGAS/DeepEval

The paper provides a comparative evaluation of Retrieval-Augmented Generation (RAG) architectures for cross-domain Large Language Model (LLM) applications.

Key areas covered include:

*   **Data Preparation:** The process involves preprocessing steps such as extracting content from HTML and PDF formats, removing boilerplate text, normalizing tables, and requiring canonical data.
*   **Performance and Evaluation:** The research aims to improve semantic recall and utilizes gauge models that demonstrate strong zero- and few-shot generation capabilities. It also focuses on generalization for tasks like summarization and translation.
*   **Technical Depth:** The work addresses complex technical aspects, including decomposing justifications into token-level details, managing the trade-off between reducing transparency and increasing compute cost, and dealing with surface sensitivity.
*   **Metrics and Reporting:** The evaluation emphasizes preserving precision and auditability while expandi